In [1]:
import pandas as pd
import plotly.express as px
import pycountry_convert as pc


In [2]:
LIFE_PATH = "/Users/tonytony/life_expectancy.csv"

print("Life Expectancy path:", LIFE_PATH)


Life Expectancy path: /Users/tonytony/life_expectancy.csv


In [3]:
life_df = pd.read_csv(LIFE_PATH, skiprows=4)

print("Shape:", life_df.shape)

life_df.head()


Shape: (266, 70)


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,Unnamed: 69
0,Aruba,ABW,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,64.049000,64.215000,64.602000,64.944000,65.303000,65.615000,...,75.540000,75.620000,75.880000,76.019000,75.406000,73.655000,76.226000,76.353000,NaN,NaN
1,Africa Eastern and Southern,AFE,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,44.169658,44.468838,44.877890,45.160583,45.535695,45.770723,...,62.167981,62.591275,63.330691,63.857261,63.766484,62.979999,64.487020,65.146154,NaN,NaN
2,Afghanistan,AFG,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,32.799000,33.291000,33.757000,34.201000,34.673000,35.124000,...,62.646000,62.406000,62.443000,62.941000,61.454000,60.417000,65.617000,66.035000,NaN,NaN
3,Africa Western and Central,AFW,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,37.779636,38.058956,38.681792,38.936918,39.194580,39.479784,...,56.392452,56.626439,57.036976,57.149847,57.364425,57.362572,57.987813,58.855722,NaN,NaN
4,Angola,AGO,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,37.933000,36.902000,37.168000,37.419000,37.704000,37.968000,...,61.619000,62.122000,62.622000,63.051000,63.116000,62.958000,64.246000,64.617000,NaN,NaN


In [4]:
life_df = life_df.drop(
    columns=[col for col in life_df.columns if str(col).startswith("Unnamed")],
    errors="ignore"
)

year_cols = [col for col in life_df.columns if str(col).isdigit()]
life_df[year_cols] = life_df[year_cols].apply(pd.to_numeric, errors="coerce")

print("Number of year columns:", len(year_cols))
print("Year range:", year_cols[0], "-", year_cols[-1])
print("Shape after cleaning:", life_df.shape)

life_df.head()


Number of year columns: 65
Year range: 1960 - 2024
Shape after cleaning: (266, 69)


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,Aruba,ABW,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,64.049000,64.215000,64.602000,64.944000,65.303000,65.615000,...,75.405000,75.540000,75.620000,75.880000,76.019000,75.406000,73.655000,76.226000,76.353000,NaN
1,Africa Eastern and Southern,AFE,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,44.169658,44.468838,44.877890,45.160583,45.535695,45.770723,...,61.713031,62.167981,62.591275,63.330691,63.857261,63.766484,62.979999,64.487020,65.146154,NaN
2,Afghanistan,AFG,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,32.799000,33.291000,33.757000,34.201000,34.673000,35.124000,...,62.270000,62.646000,62.406000,62.443000,62.941000,61.454000,60.417000,65.617000,66.035000,NaN
3,Africa Western and Central,AFW,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,37.779636,38.058956,38.681792,38.936918,39.194580,39.479784,...,56.038336,56.392452,56.626439,57.036976,57.149847,57.364425,57.362572,57.987813,58.855722,NaN
4,Angola,AGO,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,37.933000,36.902000,37.168000,37.419000,37.704000,37.968000,...,61.042000,61.619000,62.122000,62.622000,63.051000,63.116000,62.958000,64.246000,64.617000,NaN


In [5]:
SPECIAL_COUNTRY_CODES = {"XKX", "SXM"}

def is_country_code(code):
    code = str(code).strip()
    if code in SPECIAL_COUNTRY_CODES:
        return True
    try:
        alpha2 = pc.country_alpha3_to_country_alpha2(code)
        pc.country_alpha2_to_continent_code(alpha2)
        return True
    except:
        return False

life_df["Is Country"] = life_df["Country Code"].astype(str).apply(is_country_code)

print("Country / territory rows:", int(life_df["Is Country"].sum()))
print("Aggregate rows:", int((~life_df["Is Country"]).sum()))

life_df[["Country Name", "Country Code", "Is Country"]].head()


Country / territory rows: 215
Aggregate rows: 51


,Country Name,Country Code,Is Country
0,Aruba,ABW,True
1,Africa Eastern and Southern,AFE,False
2,Afghanistan,AFG,True
3,Africa Western and Central,AFW,False
4,Angola,AGO,True


In [6]:
life_geo = life_df.melt(
    id_vars=["Country Name", "Country Code", "Indicator Name", "Indicator Code", "Is Country"],
    value_vars=year_cols,
    var_name="Year",
    value_name="Life expectancy at birth, total (years)"
)

life_geo["Year"] = pd.to_numeric(life_geo["Year"], errors="coerce").astype("Int64")
life_geo["Life expectancy at birth, total (years)"] = pd.to_numeric(
    life_geo["Life expectancy at birth, total (years)"],
    errors="coerce"
)

print("Long format shape:", life_geo.shape)

life_geo.head()


Long format shape: (17290, 7)


,Country Name,Country Code,Indicator Name,Indicator Code,Is Country,Year,"Life expectancy at birth, total (years)"
0,Aruba,ABW,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,True,1960,64.049000
1,Africa Eastern and Southern,AFE,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,False,1960,44.169658
2,Afghanistan,AFG,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,True,1960,32.799000
3,Africa Western and Central,AFW,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,False,1960,37.779636
4,Angola,AGO,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,True,1960,37.933000


In [7]:
life_geo_clean = life_geo[
    (life_geo["Is Country"]) &
    (life_geo["Life expectancy at birth, total (years)"].notna())
].copy()

life_geo_clean["Year Label"] = life_geo_clean["Year"].astype(str)

print("Clean geo data shape:", life_geo_clean.shape)
print("Year range:", int(life_geo_clean["Year"].min()), "-", int(life_geo_clean["Year"].max()))
print("Number of countries/territories:", life_geo_clean["Country Code"].nunique())

life_geo_clean.head()


Clean geo data shape: (13726, 8)
Year range: 1960 - 2023
Number of countries/territories: 215


,Country Name,Country Code,Indicator Name,Indicator Code,Is Country,Year,"Life expectancy at birth, total (years)",Year Label
0,Aruba,ABW,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,True,1960,64.049,1960
2,Afghanistan,AFG,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,True,1960,32.799,1960
4,Angola,AGO,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,True,1960,37.933,1960
5,Albania,ALB,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,True,1960,56.413,1960
6,Andorra,AND,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,True,1960,72.094,1960


In [8]:
life_coverage = (
    life_geo_clean
    .groupby("Year")
    .agg(
        country_count=("Country Code", "nunique"),
        min_life_expectancy=("Life expectancy at birth, total (years)", "min"),
        median_life_expectancy=("Life expectancy at birth, total (years)", "median"),
        mean_life_expectancy=("Life expectancy at birth, total (years)", "mean"),
        max_life_expectancy=("Life expectancy at birth, total (years)", "max")
    )
    .reset_index()
)

print("Life Expectancy coverage by year:")
life_coverage.tail(15)


Life Expectancy coverage by year:


,Year,country_count,min_life_expectancy,median_life_expectancy,mean_life_expectancy,max_life_expectancy
49,2009,215,14.665,72.306000,70.333390,84.187000
50,2010,215,45.577,72.886000,70.763245,84.644000
51,2011,215,32.453,72.913000,71.099334,85.010000
52,2012,215,48.235,73.105000,71.453537,85.045000
53,2013,215,49.487,73.119512,71.749684,85.056000
54,2014,215,40.265,73.145000,71.897209,85.093000
55,2015,215,39.757,73.624390,72.105820,85.323000
56,2016,215,36.720,73.826829,72.335642,85.630000
57,2017,215,35.351,74.026000,72.492288,85.878000
58,2018,215,51.905,73.985000,72.859413,86.084000


In [10]:
year_order = [str(year) for year in sorted(life_geo_clean["Year"].dropna().astype(int).unique())]

color_min = life_geo_clean["Life expectancy at birth, total (years)"].min()
color_max = life_geo_clean["Life expectancy at birth, total (years)"].max()

life_animation = px.choropleth(
    life_geo_clean,
    locations="Country Code",
    locationmode="ISO-3",
    color="Life expectancy at birth, total (years)",
    animation_frame="Year",
    animation_group="Country Code",
    hover_name="Country Name",
    hover_data={
        "Country Code": False,
        "Year": True,
        "Life expectancy at birth, total (years)": ":.2f"
    },
    category_orders={"Year": year_order},
    color_continuous_scale="Viridis",
    range_color=[color_min, color_max],
    projection="natural earth",
    title="Life Expectancy by Year",
    labels={
        "Life expectancy at birth, total (years)": "Life Expectancy"
    }
)

life_animation.update_layout(
    height=650,
    margin=dict(l=0, r=0, t=60, b=0),
    coloraxis_colorbar=dict(title="Life Expectancy")
)

life_animation.show()


- Countries in Europe, North America, Japan, Australia, and New Zealand generally have high life expectancy.
- Many African countries have lower life expectancy in earlier periods, but most show improvement over time.
- Compared with GDP and Population, Life Expectancy has a narrower value range and fewer extreme outliers.
- Improvements in life expectancy may reflect progress in healthcare, living conditions, nutrition, and social development.
- The year 2024 has no available data in this dataset, so the animation should be interpreted only up to 2023.

Conclusion: Life Expectancy is a useful indicator of human development and shows broad improvement over time. However, differences between developed and developing regions still remain.